<a href="https://colab.research.google.com/github/Yan9595/ALBEF/blob/Test/Pretrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/ALBEF/

/content/drive/MyDrive/ALBEF


In [ ]:
# !unzip -q /content/drive/MyDrive/ALBEF/data/vqa/image/train2014.zip

In [ ]:

!pip install transformers==4.25.1
!pip install ruamel.yaml==0.17.*


In [ ]:
'''
 * Copyright (c) 2021, salesforce.com, inc.
 * All rights reserved.
 * SPDX-License-Identifier: BSD-3-Clause
 * For full license text, see LICENSE.txt file in the repo root or https://opensource.org/licenses/BSD-3-Clause
'''

import argparse
import os
import ruamel.yaml as yaml
import numpy as np
import random
import time
import datetime
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.backends.cudnn as cudnn
import torch.distributed as dist

from models.model_pretrain import ALBEF
from models.vit import interpolate_pos_embed
from models.tokenization_bert import BertTokenizer

import utils
from dataset import create_dataset, create_sampler, create_loader
from scheduler import create_scheduler
from optim import create_optimizer
%load_ext autoreload
%autoreload 2






/usr/local/lib/python3.11/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.11/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
# import os
# import zipfile
# import urllib.request

# # Define download URL and paths
# url = "http://images.cocodataset.org/zips/val2014.zip"
# output_dir = "data/vqa/image"
# zip_path = os.path.join(output_dir, "val2014.zip")

# # Create the directory if it doesn't exist
# os.makedirs(output_dir, exist_ok=True)

# # Download the file
# if not os.path.exists(zip_path):
#     print("Downloading val2014.zip...")
#     urllib.request.urlretrieve(url, zip_path)
#     print("Download complete.")
# else:
#     urllib.request.urlretrieve(url, zip_path)
#     print("val2014.zip already exists.")

# # Unzip the file
# print("Extracting val2014.zip...")
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(output_dir)
# print("Extraction complete.")


In [ ]:
def train(model, data_loader, optimizer, tokenizer, epoch, warmup_steps, device, scheduler, config):
    # train
    model.train()

    metric_logger = utils.MetricLogger(delimiter="  ")
    metric_logger.add_meter('lr', utils.SmoothedValue(window_size=50, fmt='{value:.6f}'))
    metric_logger.add_meter('loss_mlm', utils.SmoothedValue(window_size=50, fmt='{value:.4f}'))
    metric_logger.add_meter('loss_ita', utils.SmoothedValue(window_size=50, fmt='{value:.4f}'))
    metric_logger.add_meter('loss_itm', utils.SmoothedValue(window_size=50, fmt='{value:.4f}'))

    header = 'Train Epoch: [{}]'.format(epoch)
    print_freq = 50
    step_size = 100
    warmup_iterations = warmup_steps*step_size

    if args.distributed:
        data_loader.sampler.set_epoch(epoch)

    for i, (image, text) in enumerate(metric_logger.log_every(data_loader, print_freq, header)):

        optimizer.zero_grad()

        image = image.to(device,non_blocking=True)

        text_input = tokenizer(text, padding='longest', truncation=True, max_length=25, return_tensors="pt").to(device)

        if epoch>0:
            alpha = config['alpha']
        else:
            alpha = config['alpha']*min(1,i/len(data_loader))

        loss_mlm, loss_ita, loss_itm = model(image, text_input, alpha = alpha)

        loss = loss_mlm + loss_ita + loss_itm

        loss.backward()
        optimizer.step()

        metric_logger.update(loss_mlm=loss_mlm.item())
        metric_logger.update(loss_ita=loss_ita.item())
        metric_logger.update(loss_itm=loss_itm.item())
        metric_logger.update(lr=optimizer.param_groups[0]["lr"])

        if epoch==0 and i%step_size==0 and i<=warmup_iterations:
            scheduler.step(i//step_size)

    # gather the stats from all processes
    metric_logger.synchronize_between_processes()
    print("Averaged stats:", metric_logger.global_avg())
    return {k: "{:.3f}".format(meter.global_avg) for k, meter in metric_logger.meters.items()}


def main(args, config):
    utils.init_distributed_mode(args)

    device = torch.device(args.device)

    # fix the seed for reproducibility
    seed = args.seed + utils.get_rank()
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    cudnn.benchmark = True

    start_epoch = 0
    max_epoch = config['schedular']['epochs']
    warmup_steps = config['schedular']['warmup_epochs']

    #### Dataset ####
    print("Creating dataset")
    datasets = [create_dataset('pretrain', config)]

    if args.distributed:
        num_tasks = utils.get_world_size()
        global_rank = utils.get_rank()
        samplers = create_sampler(datasets, [True], num_tasks, global_rank)
    else:
        samplers = [None]

    data_loader = create_loader(datasets,samplers,batch_size=[config['batch_size']], num_workers=[4], is_trains=[True], collate_fns=[None])[0]

    tokenizer = BertTokenizer.from_pretrained(args.text_encoder)

    #### Model ####
    print("Creating model")
    model = ALBEF(config=config, text_encoder=args.text_encoder, tokenizer=tokenizer, init_deit=True)

    model = model.to(device)

    arg_opt = utils.AttrDict(config['optimizer'])
    optimizer = create_optimizer(arg_opt, model)
    arg_sche = utils.AttrDict(config['schedular'])
    lr_scheduler, _ = create_scheduler(arg_sche, optimizer)


    if args.checkpoint:
        checkpoint = torch.load(args.checkpoint, map_location='cpu')
        state_dict = checkpoint['model']
        if args.resume:
            optimizer.load_state_dict(checkpoint['optimizer'])
            lr_scheduler.load_state_dict(checkpoint['lr_scheduler'])
            start_epoch = checkpoint['epoch']+1
        else:
            pos_embed_reshaped = interpolate_pos_embed(state_dict['visual_encoder.pos_embed'],model.visual_encoder)
            m_pos_embed_reshaped = interpolate_pos_embed(state_dict['visual_encoder_m.pos_embed'],model.visual_encoder_m)
            state_dict['visual_encoder.pos_embed'] = pos_embed_reshaped
            state_dict['visual_encoder_m.pos_embed'] = m_pos_embed_reshaped
        model.load_state_dict(state_dict)
        print('load checkpoint from %s'%args.checkpoint)

    model_without_ddp = model
    if args.distributed:
        model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[args.gpu])
        model_without_ddp = model.module

    print("Start training")
    start_time = time.time()

    for epoch in range(start_epoch, max_epoch):

        if epoch>0:
            lr_scheduler.step(epoch+warmup_steps)

        train_stats = train(model, data_loader, optimizer, tokenizer, epoch, warmup_steps, device, lr_scheduler, config)
        if utils.is_main_process():
            log_stats = {**{f'train_{k}': v for k, v in train_stats.items()},
                         'epoch': epoch,
                        }
            save_obj = {
                'model': model_without_ddp.state_dict(),
                'optimizer': optimizer.state_dict(),
                'lr_scheduler': lr_scheduler.state_dict(),
                'config': config,
                'epoch': epoch,
            }
            torch.save(save_obj, os.path.join(args.output_dir, 'checkpoint_%02d.pth'%epoch))

            with open(os.path.join(args.output_dir, "log.txt"),"a") as f:
                f.write(json.dumps(log_stats) + "\n")

        dist.barrier()

    total_time = time.time() - start_time
    total_time_str = str(datetime.timedelta(seconds=int(total_time)))
    print('Training time {}'.format(total_time_str))


In [ ]:
# # import os

# # Where your COCO train2014 images are actually located
# actual_coco_path_train = "./data/vqa/image/train2014"
# actual_coco_path_val = "./data/vqa/image/val2014"


# # Path to all pretrain json files
# pretrain_dir = "./data"

# # Loop through and fix all files
# for filename in os.listdir(pretrain_dir):
#     if filename.endswith('coco.json'):
#         path = os.path.join(pretrain_dir, filename)
#         try:
#             with open(path, 'r', encoding='utf-8') as f:  # First try UTF-8
#                 data = json.load(f)
#         except UnicodeDecodeError:
#             print()

#         for item in data:
#             img_name = os.path.basename(item['image'])  # Just get "COCO_train2014_xxx.jpg"
#             if 'train' in img_name:
#                 item['image'] = os.path.join(actual_coco_path_train, img_name)
#             elif 'val' in img_name:
#                 item['image'] = os.path.join(actual_coco_path_val, img_name)

#         # Save back the fixed file (always in UTF-8)
#         with open(path, 'w', encoding='utf-8') as f:
#             json.dump(data, f)

#         print(f"✅ Fixed paths in {filename}")b

In [ ]:
# import os
# import json

# # COCO image directories
# actual_coco_path_train = "./data/vqa/image/train2014"
# actual_coco_path_val = "./data/vqa/image/val2014"

# # Directory with pretrain *.coco.json files
# pretrain_dir = "./data"

# # Process each file
# for filename in os.listdir(pretrain_dir):
#     if filename.endswith('coco_train.json'):
#         path = os.path.join(pretrain_dir, filename)

#         try:
#             with open(path, 'r', encoding='utf-8') as f:
#                 data = json.load(f)
#         except (UnicodeDecodeError, json.JSONDecodeError):
#             print(f"❌ Failed to read or decode {filename}, skipping.")
#             continue

#         if not isinstance(data, list):
#             print(f"⚠️ Skipping {filename}: expected list of dicts.")
#             continue

#         filtered_data = []
#         missing_count = 0
#         updated_count = 0

#         for item in data:
#             if not isinstance(item, dict) or 'image' not in item:
#                 continue

#             img_name = os.path.basename(item['image'])
#             if 'train' in img_name:
#                 new_path = os.path.join(actual_coco_path_train, img_name)
#             elif 'val' in img_name:
#                 new_path = os.path.join(actual_coco_path_val, img_name)
#             else:
#                 continue

#             if os.path.exists(new_path):
#                 item['image'] = new_path
#                 filtered_data.append(item)
#                 updated_count += 1
#             else:
#                 missing_count += 1

#         # Save cleaned file
#         with open(path, 'w', encoding='utf-8') as f:
#             json.dump(filtered_data, f)

#         print(f"✅ Processed {filename}: {updated_count} kept, {missing_count} missing and removed.")


KeyboardInterrupt: 

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument('--config', default='./configs/Pretrain.yaml')
parser.add_argument('--checkpoint', default='')
parser.add_argument('--resume', default=False, type=bool)
parser.add_argument('--output_dir', default='Pretrain/')
parser.add_argument('--text_encoder', default='bert-base-uncased')
parser.add_argument('--device', default='cuda')
parser.add_argument('--seed', default=42, type=int)
parser.add_argument('--world_size', default=1, type=int, help='number of distributed processes')
parser.add_argument('--dist_url', default='env://', help='url used to set up distributed training')
parser.add_argument('--distributed', default=False, type=bool)
# Instead of using parser.parse_args(), directly define the arguments:
args = argparse.Namespace(
    config='./configs/Pretrain.yaml',
    checkpoint='',
    resume=False,
    output_dir='Pretrain/',
    text_encoder='bert-base-uncased',
    device='cuda',
    seed=42,
    world_size=1,
    dist_url='env://',
    distributed=False
)

config = yaml.load(open(args.config, 'r'), Loader=yaml.Loader)

Path(args.output_dir).mkdir(parents=True, exist_ok=True)

yaml.dump(config, open(os.path.join(args.output_dir, 'config.yaml'), 'w'))

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [ ]:
utils.init_distributed_mode(args)

device = torch.device(args.device)

# fix the seed for reproducibility
seed = args.seed + utils.get_rank()
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
cudnn.benchmark = True

start_epoch = 0
max_epoch = config['schedular']['epochs']
warmup_steps = config['schedular']['warmup_epochs']

#### Dataset ####
print("Creating dataset")
datasets = [create_dataset('pretrain', config)]

if args.distributed:
    num_tasks = utils.get_world_size()
    global_rank = utils.get_rank()
    samplers = create_sampler(datasets, [True], num_tasks, global_rank)
else:
    samplers = [None]

data_loader = create_loader(datasets,samplers,batch_size=[config['batch_size']], num_workers=[4], is_trains=[True], collate_fns=[None])[0]

tokenizer = BertTokenizer.from_pretrained(args.text_encoder)

#### Model ####
print("Creating model")
model = ALBEF(config=config, text_encoder=args.text_encoder, tokenizer=tokenizer, init_deit=True)

model = model.to(device)

arg_opt = utils.AttrDict(config['optimizer'])
optimizer = create_optimizer(arg_opt, model)
arg_sche = utils.AttrDict(config['schedular'])
lr_scheduler, _ = create_scheduler(arg_sche, optimizer)


if args.checkpoint:
    checkpoint = torch.load(args.checkpoint, map_location='cpu')
    state_dict = checkpoint['model']
    if args.resume:
        optimizer.load_state_dict(checkpoint['optimizer'])
        lr_scheduler.load_state_dict(checkpoint['lr_scheduler'])
        start_epoch = checkpoint['epoch']+1
    else:
        pos_embed_reshaped = interpolate_pos_embed(state_dict['visual_encoder.pos_embed'],model.visual_encoder)
        m_pos_embed_reshaped = interpolate_pos_embed(state_dict['visual_encoder_m.pos_embed'],model.visual_encoder_m)
        state_dict['visual_encoder.pos_embed'] = pos_embed_reshaped
        state_dict['visual_encoder_m.pos_embed'] = m_pos_embed_reshaped
    model.load_state_dict(state_dict)
    print('load checkpoint from %s'%args.checkpoint)

model_without_ddp = model
if args.distributed:
    model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[args.gpu])
    model_without_ddp = model.module

print("Start training")
start_time = time.time()

for epoch in range(start_epoch, max_epoch):

    if epoch>0:
        lr_scheduler.step(epoch+warmup_steps)

    train_stats = train(model, data_loader, optimizer, tokenizer, epoch, warmup_steps, device, lr_scheduler, config)

    log_stats = {**{f'train_{k}': v for k, v in train_stats.items()},
                  'epoch': epoch,
                }
    save_obj = {
        'model': model_without_ddp.state_dict(),
        'optimizer': optimizer.state_dict(),
        'lr_scheduler': lr_scheduler.state_dict(),
        'config': config,
        'epoch': epoch,
    }
    torch.save(save_obj, os.path.join(args.output_dir, 'checkpoint_%02d.pth'%epoch))

    with open(os.path.join(args.output_dir, "log.txt"),"a") as f:
        f.write(json.dumps(log_stats) + "\n")

   #dist.barrier()

total_time = time.time() - start_time
total_time_str = str(datetime.timedelta(seconds=int(total_time)))
print('Training time {}'.format(total_time_str))


Not using distributed mode
Creating dataset


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Creating model
reshape position embedding from 196 to 16
_IncompatibleKeys(missing_keys=[], unexpected_keys=['head.weight', 'head.bias'])
Start training


/content/drive/MyDrive/ALBEF/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale
/content/drive/MyDrive/ALBEF/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale
/content/drive/MyDrive/ALBEF/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale
/content/drive/MyDrive/ALBEF/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale


Train Epoch: [0]  [   0/1862]  eta: 2:14:22  lr: 0.000010  loss_mlm: 7.6358  loss_ita: 14.4293  loss_itm: 0.7124  time: 4.3301  data: 2.6602  max mem: 11103
Train Epoch: [0]  [  50/1862]  eta: 3:19:32  lr: 0.000010  loss_mlm: 4.9611  loss_ita: 9.1312  loss_itm: 0.6463  time: 6.8985  data: 6.2192  max mem: 13039
Train Epoch: [0]  [ 100/1862]  eta: 3:05:59  lr: 0.000010  loss_mlm: 4.6894  loss_ita: 9.1231  loss_itm: 0.6412  time: 6.1276  data: 5.4313  max mem: 13039
Train Epoch: [0]  [ 150/1862]  eta: 2:51:10  lr: 0.000015  loss_mlm: 4.4242  loss_ita: 9.2882  loss_itm: 0.6385  time: 5.4112  data: 4.7141  max mem: 13039
Train Epoch: [0]  [ 200/1862]  eta: 2:37:56  lr: 0.000015  loss_mlm: 4.0212  loss_ita: 9.1208  loss_itm: 0.6367  time: 4.6402  data: 3.9524  max mem: 13039
Train Epoch: [0]  [ 250/1862]  eta: 2:24:08  lr: 0.000019  loss_mlm: 4.2202  loss_ita: 8.6575  loss_itm: 0.6361  time: 4.5101  data: 3.8112  max mem: 13039
Train Epoch: [0]  [ 300/1862]  eta: 2:14:00  lr: 0.000019  loss

ValueError: Default process group has not been initialized, please make sure to call init_process_group.

In [ ]:
epoch

0

In [ ]:
epoch

0

In [ ]:
log_stats = {**{f'train_{k}': v for k, v in train_stats.items()},
                      'epoch': epoch,
                    }
save_obj = {
    'model': model_without_ddp.state_dict(),
    'optimizer': optimizer.state_dict(),
    'lr_scheduler': lr_scheduler.state_dict(),
    'config': config,
    'epoch': epoch,
}
torch.save(save_obj, os.path.join(args.output_dir, 'checkpoint_%02d.pth'%epoch))

with open(os.path.join(args.output_dir, "log.txt"),"a") as f:
    f.write(json.dumps(log_stats) + "\n")


NameError: name 'train_stats' is not defined

In [ ]:
missing_files = [ann['image'] for ann in datasets[0].ann if not os.path.exists(ann['image'])]
print(f"{len(missing_files)} missing images detected.")


142236 missing images detected.


In [ ]:
import os
import urllib.request
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

def download_image(img_path):
    base_url = "http://images.cocodataset.org/val2014/"
    filename = os.path.basename(img_path)
    url = base_url + filename

    os.makedirs(os.path.dirname(img_path), exist_ok=True)
    try:
        urllib.request.urlretrieve(url, img_path)
        return (img_path, True)
    except Exception as e:
        return (img_path, False, str(e))


# Set the number of threads (adjust based on your bandwidth/CPU)
max_workers = 32

# Start multithreaded download
results = []
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(download_image, img_path) for img_path in missing_files]
    for f in tqdm(as_completed(futures), total=len(futures), desc="Downloading missing images"):
        results.append(f.result())

# Optional: log failed downloads
failed = [res for res in results if res[1] is False]
if failed:
    print(f"\n{len(failed)} downloads failed:")
    for f in failed:
        print(f"[Failed] {f[0]}: {f[2]}")


KeyboardInterrupt: 

In [ ]:
if __name__ == '__main__':
  main(args, config)

Not using distributed mode
Creating dataset


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Creating model
reshape position embedding from 196 to 64
_IncompatibleKeys(missing_keys=[], unexpected_keys=['head.weight', 'head.bias'])
Start training


/content/drive/MyDrive/ALBEF/dataset/randaugment.py:31: RuntimeWarning: overflow encountered in scalar negative
  offset = -low * scale


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/fetch.py", line 52, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/content/drive/MyDrive/ALBEF/dataset/caption_dataset.py", line 106, in __getitem__
    image = Image.open(ann['image']).convert('RGB')
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/PIL/Image.py"

In [ ]:
import os

# Target directory (create if needed)
PRETRAIN_DATA_DIR = "./data"
os.makedirs(PRETRAIN_DATA_DIR, exist_ok=True)

# Define URL and target path
url = "https://storage.googleapis.com/sfr-pcl-data-research/ALBEF/json_pretrain.zip"
zip_path = os.path.join(PRETRAIN_DATA_DIR, "json_pretrain.zip")

# Download the file
!wget -O "{zip_path}" "{url}"

# Extract the contents in place (flatten structure)
!unzip -j "{zip_path}" -d "{PRETRAIN_DATA_DIR}"

# Delete the ZIP file
os.remove(zip_path)

# List the extracted files
for f in os.listdir(PRETRAIN_DATA_DIR):
    print("✅", f)


--2025-04-20 21:21:34--  https://storage.googleapis.com/sfr-pcl-data-research/ALBEF/json_pretrain.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 172.253.118.207, 74.125.200.207, 74.125.130.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.253.118.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 660749965 (630M) [application/zip]
Saving to: ‘./data/json_pretrain.zip’

./data/json_pretrai 100%[===================>] 630.14M  20.2MB/s    in 30s     

2025-04-20 21:22:04 (21.3 MB/s) - ‘./data/json_pretrain.zip’ saved [660749965/660749965]

Archive:  ./data/json_pretrain.zip
  inflating: ./data/cc12m.json       
  inflating: ./data/._cc12m.json     
  inflating: ./data/coco.json        
  inflating: ./data/._coco.json      
  inflating: ./data/vg.json          
  inflating: ./data/._vg.json        
  inflating: ./data/sbu.json         
  inflating: ./data/._sbu.json       
  inflating: ./data/cc3m_val.json    
  inflatin

In [ ]:
%%python -m torch.distributed.launch --nproc_per_node=8 --use_env Pretrain.py --config ./configs/Pretrain.yaml --output_dir output/Pretrain

| distributed init (rank 0): env://


/usr/local/lib/python3.11/dist-packages/torch/distributed/launch.py:208: FutureWarning: The module torch.distributed.launch is deprecated
and will be removed in future. Use torchrun.
Note that --use-env is set by default in torchrun.
If your script expects `--local-rank` argument to be set, please
change it to read from `os.environ['LOCAL_RANK']` instead. See 
https://pytorch.org/docs/stable/distributed.html#launch-utility for 
further instructions

  main()
W0420 21:13:50.179000 4498 torch/distributed/run.py:793] 
W0420 21:13:50.179000 4498 torch/distributed/run.py:793] *****************************************
W0420 21:13:50.179000 4498 torch/distributed/run.py:793] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0420 21:13:50.179000 4498 torch/distributed/run.py:793] *****************************************
/usr/local/lib

CalledProcessError: Command 'b' \n'' returned non-zero exit status 1.